<a href="https://colab.research.google.com/github/zencolab/WhatDreamsCost-ComfyUI/blob/main/IndexTTS_2_5_Colab_L4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IndexTTS 2.5：免费生成SRT并自动换声

**从上到下依次运行即可。**只需一次上传两个文件：

1. 剪映导出的视频（MP4/MOV/MKV/WebM）
2. 已获授权的人物参考声音（WAV/MP3/M4A/FLAC，建议10～30秒、单人、无音乐）

程序使用开源免费的 Faster-Whisper 自动生成中文字幕；之后可修改错句、错词或在空白时间增加短暂台词，最终输出 `/content/video_new_voice.mp4`。

> 当前版本会删除视频全部旧音频，只保留新人物声音；背景音乐和音效也会被删除。


In [ ]:
# 步骤1：确认使用 Colab Pro 的 NVIDIA L4 GPU
import os, subprocess
assert os.path.exists('/content'), '请在 Google Colab 中运行。'
try:
    gpu = subprocess.check_output(
        ['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], text=True
    ).strip()
except Exception as exc:
    raise RuntimeError('未检测到 NVIDIA GPU，请在运行时类型中选择 L4。') from exc
print('检测到：', gpu)
assert 'L4' in gpu, f'当前不是 L4：{gpu}'
print('✅ L4 GPU 可用')


In [ ]:
# 步骤2：安装免费字幕识别、IndexTTS 2.5和辅助脚本
# 第一次运行时间较长；下载中断时重新运行本格。
%cd /content
!python -m pip install -q -U uv huggingface_hub hf_xet pysrt faster-whisper

from pathlib import Path
from huggingface_hub import snapshot_download
import subprocess, urllib.request

repo = Path('/content/index-tts')
if not repo.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/index-tts/index-tts.git',str(repo)], check=True)
else:
    subprocess.run(['git','-C',str(repo),'fetch','--depth','1','origin','main'], check=True)
    subprocess.run(['git','-C',str(repo),'reset','--hard','origin/main'], check=True)

%cd /content/index-tts
!uv sync --extra webui
!uv pip install --python /content/index-tts/.venv/bin/python pysrt pydub

model_dir = Path('/content/index-tts/checkpoints')
model_dir.mkdir(parents=True, exist_ok=True)
snapshot_download(repo_id='IndexTeam/IndexTTS-2.5', local_dir=str(model_dir))
assert (model_dir/'config.yaml').exists(), 'IndexTTS模型下载不完整。'

urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/zencolab/WhatDreamsCost-ComfyUI/main/auto_replace_voice.py',
    '/content/index-tts/auto_replace_voice.py',
)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/zencolab/WhatDreamsCost-ComfyUI/main/generate_srt.py',
    '/content/generate_srt.py',
)
print('✅ 安装和下载完成')


## 步骤3：上传视频和参考声音，免费生成SRT

同时选择两个文件。Faster-Whisper首次运行会下载模型，完成后自动显示字幕编号。


In [ ]:
# 步骤3：上传两个文件并免费生成中文字幕
from google.colab import files
from pathlib import Path
import subprocess, sys, pysrt

uploaded = files.upload()
names = list(uploaded)

def one(exts, label):
    matches = [n for n in names if Path(n).suffix.lower() in exts]
    if len(matches) != 1:
        raise ValueError(f'需要且只能上传一个{label}；识别到：{matches}')
    return matches[0]

video = one({'.mp4','.mov','.mkv','.webm'}, '视频')
voice = one({'.wav','.mp3','.m4a','.flac','.ogg'}, '参考声音')
Path('/content/video.mp4').write_bytes(uploaded[video])
voice_source = Path('/content/uploaded_voice' + Path(voice).suffix.lower())
voice_source.write_bytes(uploaded[voice])
subprocess.run([
    'ffmpeg','-y','-loglevel','error','-i',str(voice_source),
    '-ac','1','-ar','24000','/content/voice.wav'
], check=True)

# 单独进程识别，结束后会释放Whisper占用的显存。
subprocess.run([
    sys.executable, '/content/generate_srt.py',
    '--video','/content/video.mp4',
    '--output','/content/subtitles.srt',
    '--model','large-v3', '--language','zh',
], check=True)

print('✅ 免费SRT已生成。字幕如下：')
for row in pysrt.open('/content/subtitles.srt', encoding='utf-8'):
    print(f'#{row.index}  {row.start} --> {row.end}  |  {row.text.replace(chr(10), " ")}')


## 步骤4：修改或增加台词（可选）

- 修改错句/错词：在 `CORRECTIONS` 填字幕编号和新文字。
- 增加台词：在 `ADDITIONS` 填开始时间、结束时间和文字，必须使用原来无人说话的空白时间。


In [ ]:
# 步骤4：填写修改；不修改时保持为空并运行本格
CORRECTIONS = {
    # 3: '这里填写第3条字幕的正确台词',
}
ADDITIONS = [
    # {'start':'00:00:12,500', 'end':'00:00:13,500', 'text':'这里填写新增台词'},
]

import json
from pathlib import Path
Path('/content/dialogue_edits.json').write_text(
    json.dumps({'corrections':CORRECTIONS, 'additions':ADDITIONS}, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(f'✅ 已保存：修正 {len(CORRECTIONS)} 条，新增 {len(ADDITIONS)} 条')


In [ ]:
# 步骤5：自动生成新声音并替换视频原音轨
import os, subprocess
env = os.environ.copy()
env['PYTHONPATH'] = '/content/index-tts' + os.pathsep + env.get('PYTHONPATH','')
subprocess.run([
    '/content/index-tts/.venv/bin/python',
    '/content/index-tts/auto_replace_voice.py',
], cwd='/content/index-tts', env=env, check=True)
print('✅ 输出：/content/video_new_voice.mp4')


In [ ]:
# 步骤6：预览并下载成品
from IPython.display import Video, display
from google.colab import files
output = '/content/video_new_voice.mp4'
display(Video(output, embed=True))
files.download(output)
